In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path

DRIVE_REPO = Path('/content/drive/MyDrive/movie-recomender-search')
LOCAL_REPO = Path('/content/movie-recomender-search')

if LOCAL_REPO.exists():
    shutil.rmtree(LOCAL_REPO)

shutil.copytree(DRIVE_REPO, LOCAL_REPO)
os.chdir(LOCAL_REPO)

print('Working directory:', Path.cwd())
print('Drive repo:', DRIVE_REPO)
print('Local repo:', LOCAL_REPO)

# 01 - EDA MovieLens 1M

Mục tiêu notebook này:
- đọc bộ dữ liệu MovieLens 1M chính thức
- kiểm tra quy mô và độ thưa của dữ liệu
- quan sát phân bố rating và mức độ tương tác
- chuẩn bị cơ sở cho bước huấn luyện MF và NeuMF

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.load_movielens import load_movielens_1m

sns.set_theme(style='whitegrid')
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
users, movies, ratings = load_movielens_1m()

n_users = int(ratings['user_id'].nunique())
n_items = int(ratings['movie_id'].nunique())
n_interactions = int(len(ratings))
density = n_interactions / (n_users * n_items)

summary = pd.Series(
    {
        'users_rows': len(users),
        'movies_rows': len(movies),
        'ratings_rows': n_interactions,
        'unique_users_in_ratings': n_users,
        'unique_items_in_ratings': n_items,
        'matrix_density': round(density, 4),
        'rating_min': int(ratings['rating'].min()),
        'rating_max': int(ratings['rating'].max()),
    }
)
summary

In [ ]:
display(users.head())
display(movies.head())
display(ratings.head())

In [ ]:
ratings_per_user = ratings.groupby('user_id').size()
ratings_per_item = ratings.groupby('movie_id').size()

interaction_summary = pd.DataFrame(
    {
        'metric': ['ratings_per_user', 'ratings_per_item'],
        'min': [ratings_per_user.min(), ratings_per_item.min()],
        'median': [ratings_per_user.median(), ratings_per_item.median()],
        'mean': [ratings_per_user.mean(), ratings_per_item.mean()],
        'max': [ratings_per_user.max(), ratings_per_item.max()],
    }
)
interaction_summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ratings['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='#4c72b0')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')

ratings_per_user.hist(ax=axes[1], bins=40, color='#55a868')
axes[1].set_title('Ratings per User')
axes[1].set_xlabel('Count')

ratings_per_item.hist(ax=axes[2], bins=40, color='#c44e52')
axes[2].set_title('Ratings per Item')
axes[2].set_xlabel('Count')

plt.tight_layout()
plt.show()